In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("pipeline_run_id", "")

from pyspark.sql.functions import col, upper

pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

print("=" * 80)
print("ENTERPRISE GENERIC GOLD LOADER V4")
print("=" * 80)
print("Pipeline Run :", pipeline_run_id)

ENTERPRISE GENERIC GOLD LOADER V4
Pipeline Run : 


In [0]:
NOTEBOOK_NAME = "Enterprise_Generic_Gold_Loader_v4"

def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def print_info(name, value):
    print(f"{name:<30}: {value}")

def print_success(message):
    print(f"SUCCESS : {message}")

def print_warning(message):
    print(f"WARNING : {message}")

def print_error(message):
    print(f"ERROR   : {message}")

In [0]:
print_header("READING ALL ACTIVE METADATA")

# Only tables that have a gold_table defined actually get a Gold layer.
metadata_rows = (
    spark.table(f"{catalog_name}.{metadata_schema}.bronze_config")
         .filter(upper(col("active")) == "Y")
         .orderBy("table_name")
         .collect()
)

metadata_rows = [
    r for r in metadata_rows
    if r["gold_table"] is not None and str(r["gold_table"]).strip() != ""
]

if len(metadata_rows) == 0:
    raise Exception("No active tables with a gold_table defined were found in bronze_config.")

print_success(f"{len(metadata_rows)} active table(s) with Gold targets found.")
for r in metadata_rows:
    print_info("Active Table", f"{r['table_name']} -> {r['gold_table']}")


READING ALL ACTIVE METADATA
SUCCESS : 5 active table(s) with Gold targets found.
Active Table                  : Agent -> dim_agent
Active Table                  : Branch -> dim_branch
Active Table                  : Claim -> fact_claim
Active Table                  : Customer -> dim_customer
Active Table                  : Policy -> dim_policy


In [0]:
print_header("PROCESSING ALL ACTIVE TABLES")

results_summary = []

for config in metadata_rows:

    table_name = config["table_name"]

    print_header(f"PROCESSING {table_name}")

    try:
        # ---------------- Metadata ----------------
        target_table  = config["target_table"]
        gold_table    = config["gold_table"]
        load_strategy = config["load_strategy"]

        if target_table is None or str(target_table).strip() == "":
            raise Exception("Mandatory metadata field 'target_table' is missing.")
        if gold_table is None or str(gold_table).strip() == "":
            raise Exception("Mandatory metadata field 'gold_table' is missing.")

        silver_table = f"{catalog_name}.{silver_schema}.{target_table}"
        gold_table_name = f"{catalog_name}.{gold_schema}.{gold_table}"

        print_info("Silver Table", silver_table)
        print_info("Gold Table", gold_table_name)

        # ---------------- Read Silver ----------------
        if not spark.catalog.tableExists(silver_table):
            raise Exception(f"Silver table does not exist yet : {silver_table}")

        df = spark.table(silver_table)
        rows_read = df.count()
        print_info("Rows Read (Silver)", rows_read)

        # ---------------- Write Gold ----------------
        # Gold always reflects the FULL current state, rebuilt from Silver on every run —
        # not append. Silver already holds the deduplicated current snapshot, so Gold is
        # a full overwrite regardless of load_strategy (FULL vs INCREMENTAL governs Bronze
        # ingestion only, not how Gold is materialized).
        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(gold_table_name)
        )

        rows_written = rows_read
        gold_total_rows = spark.table(gold_table_name).count()

        print_success("Gold Load Completed")
        print_info("Rows Written", rows_written)
        print_info("Gold Total Rows", gold_total_rows)

        # ---------------- Audit ----------------
        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=gold_table_name,
            load_type=load_strategy,
            rows_read=rows_read,
            rows_written=rows_written,
            status="SUCCESS",
            error_message="",
            pipeline_run_id=pipeline_run_id
        )
        print_success("Audit Written")

        results_summary.append({
            "table_name": table_name,
            "status": "SUCCESS",
            "rows_read": rows_read,
            "rows_written": rows_written,
            "error": ""
        })

    except Exception as ex:

        print_error(str(ex))

        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=table_name,
            load_type=config["load_strategy"] if config["load_strategy"] else "UNKNOWN",
            rows_read=0,
            rows_written=0,
            status="FAILED",
            error_message=str(ex),
            pipeline_run_id=pipeline_run_id
        )

        results_summary.append({
            "table_name": table_name,
            "status": "FAILED",
            "rows_read": 0,
            "rows_written": 0,
            "error": str(ex)
        })

        continue


PROCESSING ALL ACTIVE TABLES

PROCESSING Agent
Silver Table                  : dbw_insurance.insurance_silver.agent
Gold Table                    : dbw_insurance.insurance_gold.dim_agent
Rows Read (Silver)            : 1000
SUCCESS : Gold Load Completed
Rows Written                  : 1000
Gold Total Rows               : 1000
SUCCESS : Audit Written

PROCESSING Branch
Silver Table                  : dbw_insurance.insurance_silver.branch
Gold Table                    : dbw_insurance.insurance_gold.dim_branch
Rows Read (Silver)            : 1000
SUCCESS : Gold Load Completed
Rows Written                  : 1000
Gold Total Rows               : 1000
SUCCESS : Audit Written

PROCESSING Claim
Silver Table                  : dbw_insurance.insurance_silver.claim
Gold Table                    : dbw_insurance.insurance_gold.fact_claim
Rows Read (Silver)            : 1000
SUCCESS : Gold Load Completed
Rows Written                  : 1000
Gold Total Rows               : 1000
SUCCESS : Audit Writt

In [0]:
print("\n")
print("=" * 90)
print("ENTERPRISE GENERIC GOLD LOADER V4 COMPLETED")
print("=" * 90)

for r in results_summary:
    line = f"{r['table_name']:<15} : {r['status']:<8} Read={r['rows_read']:<6} Written={r['rows_written']:<6}"
    if r["status"] == "FAILED":
        line += f"  ERROR: {r['error']}"
    print(line)

success_count = len([r for r in results_summary if r["status"] == "SUCCESS"])
failed_count = len([r for r in results_summary if r["status"] == "FAILED"])

print("=" * 90)
print_success(f"Processed {len(results_summary)} tables. Success={success_count}  Failed={failed_count}")
print("=" * 90)



ENTERPRISE GENERIC GOLD LOADER V4 COMPLETED
Agent           : SUCCESS  Read=1000   Written=1000  
Branch          : SUCCESS  Read=1000   Written=1000  
Claim           : SUCCESS  Read=1000   Written=1000  
Customer        : SUCCESS  Read=1958   Written=1958  
Policy          : SUCCESS  Read=1000   Written=1000  
SUCCESS : Processed 5 tables. Success=5  Failed=0
